<a href="https://colab.research.google.com/github/SridharS-Square/Agentic_AI_Workshop/blob/main/create%20an%20Agent%20using%20LLM%20and%20custom%20mathematical%20functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
!pip install langgraph langchain google-generativeai


In [40]:
import google.generativeai as genai

from google.colab import userdata
genai.configure(api_key='')

gemini_model = genai.GenerativeModel("gemini-1.5-flash-latest")

In [41]:
# Math functions
def plus(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "Error: Division by zero."


In [42]:
from typing import TypedDict

class AgentState(TypedDict):
    input: str
    result: any

def math_node(state):
    query = state["input"]
    tokens = query.lower().split()
    try:
        a, op, b = float(tokens[0]), tokens[1], float(tokens[2])
    except:
        return {"result": "Invalid input format. Use: number operator number."}

    if "plus" in op or "add" in op:
        result = plus(a, b)
    elif "subtract" in op or "minus" in op:
        result = subtract(a, b)
    elif "multiply" in op or "times" in op:
        result = multiply(a, b)
    elif "divide" in op or "divided" in op:
        result = divide(a, b)
    else:
        result = "Operation not recognized."

    return {"result": result}

def general_node(state):
    query = state["input"]
    response = gemini_model.generate_content(query)
    return {"result": response.text}


In [43]:
def detect_task(state):
    query = state.get("input", "")
    math_keywords = ["plus", "add", "subtract", "minus", "multiply", "times", "divide", "divided"]
    if any(word in query.lower() for word in math_keywords):
        return "math"
    else:
        return "general"


In [44]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(AgentState)

# Add nodes
graph.add_node("math", math_node)
graph.add_node("general", general_node)

# Define conditional routing directly from START
graph.add_conditional_edges(START, detect_task, {
    "math": "math",
    "general": "general"
})

# End the flow
graph.add_edge("math", END)
graph.add_edge("general", END)

# Compile graph
app = graph.compile()


In [47]:
# Math Query
result = app.invoke(AgentState(input="5 plus 3", result=None))
print("Math Result:", result)

# General Query
result = app.invoke(AgentState(input="Who is the prime minister of India?", result=None))
print("General Answer:", result)


Math Result: {'input': '5 plus 3', 'result': 8.0}
General Answer: {'input': 'Who is the prime minister of India?', 'result': 'The current prime minister of India is Narendra Modi.\n'}
